# Lesson 8: Inside the Transformer — Q, K, V, positions and multiple heads 🔬
**Learning goals**
1. Calculate **self-attention** step by step with **Query, Key and Value**
2. Understand why Transformers need **positional encoding**
3. See why **multi-head attention** uses several "readers" at once
4. Build and train a **mini GPT** and compare it with last lesson's RNNs
5. Know the difference between **encoder** (BERT) and **decoder** (GPT) models

**⏱️ Timing:** Q/K/V 15' | positions 10' | multi-head 5' | mini GPT 20' | encoder vs decoder 10'

> ⚙️ GPU recommended: **Runtime → Change runtime type → T4 GPU**.

## 📚 An analogy: searching a library
In Lesson 7, each word compared itself directly with every other word. Real Transformers split each word into **three roles**:
| Role | Library analogy | In the sentence |
|---|---|---|
| **Query (Q)** | What I'm **looking for** ("books about dragons") | "it" asks: *which thing am I referring to?* |
| **Key (K)** | The **label on each book's spine** | "trophy" advertises: *I'm a big object* |
| **Value (V)** | The **content inside** the book | the information "trophy" actually passes along |

Attention score = how well my **Query** matches your **Key**. Then I collect a mix of everyone's **Values**, weighted by those scores.

**Self-attention formula:** `Attention(Q, K, V) = softmax(Q·Kᵀ / √d) · V`

## Part 1: Self-attention, step by step 🧮

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

words = ["the", "trophy", "did", "not", "fit", "because", "it", "was", "big"]
d = 8                                             # size of each word vector
X = np.random.randn(len(words), d)                # word embeddings (normally learned; random here)

# Three weight matrices turn each word into its Query, Key and Value (normally learned during training)
W_q, W_k, W_v = (np.random.randn(d, d) for _ in range(3))
Q, K, V = X @ W_q, X @ W_k, X @ W_v
print("Q, K, V shapes:", Q.shape, K.shape, V.shape, "→ one row per word")

In [ ]:
def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

scores = Q @ K.T                     # step 1: every Query vs every Key
scaled = scores / np.sqrt(d)         # step 2: scale down so softmax isn't too extreme
weights = softmax(scaled)            # step 3: turn each row into percentages
output = weights @ V                 # step 4: mix the Values

i = words.index("it")
print("How 'it' spreads its attention:")
for w, a in zip(words, weights[i]):
    print(f"  {w:8} {'█' * int(a * 40)} {a:.0%}")
print("\nEach row adds up to:", weights.sum(axis=1).round(2))

In [ ]:
plt.imshow(weights, cmap="Blues")
plt.xticks(range(len(words)), words, rotation=45); plt.yticks(range(len(words)), words)
plt.xlabel("Key (being looked at)"); plt.ylabel("Query (looking)"); plt.colorbar(); plt.show()

### 🧩 Exercise 1
1. The weights are random, so the pattern is meaningless for now. In a trained model, what pattern would you **hope** to see for the row "it"?
2. **Why divide by √d?** Remove `/ np.sqrt(d)` and look at the attention for "it". What happens? (Hint: softmax of very big numbers puts almost 100% on one word, which makes learning harder.)

## Part 2: Positional encoding — "who bit whom?" 📍
Attention compares **every word with every word at the same time**. Nothing in it says which word came **first**! Watch:

In [ ]:
sent_a = ["dog", "bites", "man"]
sent_b = ["man", "bites", "dog"]
emb = {w: np.random.randn(d) for w in ["dog", "bites", "man"]}

def attend(sentence, add_position=False):
    X = np.array([emb[w] for w in sentence])
    if add_position:
        X = X + positional_encoding(len(sentence), d)
    A = softmax((X @ W_q) @ (X @ W_k).T / np.sqrt(d)) @ (X @ W_v)
    return {w: A[j].round(2)[:3] for j, w in enumerate(sentence)}   # show first 3 numbers per word

print("A:", attend(sent_a)["dog"], "← 'dog' in 'dog bites man'")
print("B:", attend(sent_b)["dog"], "← 'dog' in 'man bites dog'")
print("Identical! Without positions, the model can't tell who did the biting.")

**The fix:** add a unique **position signal** to each word's vector. The original Transformer used waves (sine and cosine) of different speeds — like a clock with a second hand, minute hand and hour hand. Every position gets a different combination.

In [ ]:
def positional_encoding(n_positions, d):
    pos = np.arange(n_positions)[:, None]
    i = np.arange(d)[None, :]
    angle = pos / (10000 ** ((2 * (i // 2)) / d))
    return np.where(i % 2 == 0, np.sin(angle), np.cos(angle))

plt.figure(figsize=(8, 4))
plt.imshow(positional_encoding(50, 64), cmap="RdBu", aspect="auto")
plt.xlabel("vector dimension (fast waves → slow waves)"); plt.ylabel("position in sentence"); plt.colorbar()
plt.title("every row (position) has a unique pattern"); plt.show()

print("A:", attend(sent_a, add_position=True)["dog"])
print("B:", attend(sent_b, add_position=True)["dog"])
print("Different now! ✅")

## Part 3: Multi-head attention — many readers 👥
### 🕵️ Analogy: a team of detectives
One detective looks for **who** did it, another for **when**, another for **where**. Then they share notes.
Multi-head attention runs several attention calculations **in parallel**, each with its own Q, K, V weights, so each "head" can specialise (grammar, meaning, pronouns…).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
X = np.random.randn(len(words), d)
for h, ax in enumerate(axes):
    Wq, Wk = np.random.randn(d, d), np.random.randn(d, d)      # each head has its own weights
    A = softmax((X @ Wq) @ (X @ Wk).T / np.sqrt(d))
    ax.imshow(A, cmap="Blues"); ax.set_title(f"head {h + 1}")
    ax.set_xticks(range(len(words))); ax.set_xticklabels(words, rotation=60)
    ax.set_yticks(range(len(words))); ax.set_yticklabels(words)
plt.show()
print("Each head looks at the sentence differently. GPT-2 small has 12 heads in each of its 12 layers = 144 heads!")

## Part 4: Build and train a mini GPT 🛠️
A Transformer **block** = multi-head attention + a small neural network (Lesson 4), with two helpers:
- **Residual connections:** add the input back to the output ("keep your original notes, just add new ones")
- **Layer normalisation:** keep numbers in a sensible range

For **writing** text we need one more rule — a **causal mask**: when predicting word 5, the model may only look at words 1–4. **No peeking at the answer!** (Like covering the rest of the page with your hand.)

In [ ]:
L = 6
mask = np.tril(np.ones((L, L)))
plt.imshow(mask, cmap="Greens"); plt.title("causal mask: row = current position, green = allowed to look")
plt.xlabel("position looked at"); plt.ylabel("current position"); plt.show()

In [ ]:
import torch, torch.nn as nn, time
device = "cuda" if torch.cuda.is_available() else "cpu"

text = """Twinkle, twinkle, little star, how I wonder what you are. Up above the world so high, like a diamond in the sky.
Mary had a little lamb, its fleece was white as snow. And everywhere that Mary went, the lamb was sure to go.
Humpty Dumpty sat on a wall. Humpty Dumpty had a great fall. All the king's horses and all the king's men could not put Humpty together again.
Jack and Jill went up the hill to fetch a pail of water. Jack fell down and broke his crown, and Jill came tumbling after.
Hey diddle diddle, the cat and the fiddle, the cow jumped over the moon. The little dog laughed to see such sport, and the dish ran away with the spoon.
Row, row, row your boat, gently down the stream. Merrily, merrily, merrily, merrily, life is but a dream.
Baa, baa, black sheep, have you any wool? Yes sir, yes sir, three bags full. One for the master, one for the dame, and one for the little boy who lives down the lane.
Little Bo Peep has lost her sheep and does not know where to find them. Leave them alone and they will come home, wagging their tails behind them.
Hickory dickory dock, the mouse ran up the clock. The clock struck one, the mouse ran down. Hickory dickory dock.
Rain, rain, go away, come again another day. Little Johnny wants to play.
Old Mother Hubbard went to the cupboard to fetch her poor dog a bone. When she got there the cupboard was bare, and so the poor dog had none.
Little Miss Muffet sat on a tuffet, eating her curds and whey. Along came a spider who sat down beside her and frightened Miss Muffet away.
Little Jack Horner sat in the corner, eating his Christmas pie. He put in his thumb and pulled out a plum and said, what a good boy am I.
The itsy bitsy spider went up the water spout. Down came the rain and washed the spider out. Out came the sun and dried up all the rain, and the itsy bitsy spider went up the spout again.""".lower()
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], device=device)
CONTEXT = 64        # how many characters the model can see at once
print(len(text), "characters, device =", device)

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab, d=128, heads=4, layers=3):
        super().__init__()
        self.tok = nn.Embedding(vocab, d)          # what the character is
        self.pos = nn.Embedding(CONTEXT, d)        # where it is (a learned positional encoding)
        block = nn.TransformerEncoderLayer(d, heads, dim_feedforward=4 * d, dropout=0.1,
                                           batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(block, layers, enable_nested_tensor=False)
        self.out = nn.Linear(d, vocab)

    def forward(self, x):
        T = x.shape[1]
        h = self.tok(x) + self.pos(torch.arange(T, device=x.device))
        causal = nn.Transformer.generate_square_subsequent_mask(T).to(x.device)   # no peeking!
        return self.out(self.blocks(h, mask=causal, is_causal=True))

torch.manual_seed(0)
gpt = MiniGPT(len(chars)).to(device)
print(f"MiniGPT has {sum(p.numel() for p in gpt.parameters()):,} weights")

In [ ]:
opt = torch.optim.AdamW(gpt.parameters(), lr=1e-3)
losses, start = [], time.time()
for step in range(600):
    idx = torch.randint(0, len(data) - CONTEXT - 1, (32,))
    x = torch.stack([data[i:i + CONTEXT] for i in idx])
    y = torch.stack([data[i + 1:i + CONTEXT + 1] for i in idx])
    loss = nn.functional.cross_entropy(gpt(x).reshape(-1, len(chars)), y.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
    if step % 100 == 0:
        print(f"step {step:3}  loss {loss.item():.2f}")
print(f"done in {time.time() - start:.0f}s")
plt.plot(np.convolve(losses, np.ones(20) / 20, mode="valid")); plt.xlabel("step"); plt.ylabel("loss"); plt.show()

In [ ]:
def write(start="hickory", length=200, temperature=0.8):
    gpt.eval()
    ids = [stoi[c] for c in start]
    with torch.no_grad():
        for _ in range(length):
            x = torch.tensor([ids[-CONTEXT:]], device=device)       # only the last CONTEXT characters fit
            probs = torch.softmax(gpt(x)[0, -1] / temperature, dim=0)
            ids.append(torch.multinomial(probs, 1).item())
    gpt.train()
    return "".join(chars[i] for i in ids)

print(write())

### 🧩 Exercise 2: Transformer vs RNN
1. Compare this output with the LSTM and GRU from Lesson 6. Which is better? (With such a tiny dataset, both may copy the rhymes.)
2. The RNN read characters **one by one**; MiniGPT looks at **64 at once**. Why does that make Transformers much faster to train on huge datasets with GPUs?
3. Change `heads=4` to `heads=1` and `layers=3` to `layers=1`, retrain, and compare the loss.

### 🚀 Challenge: watch it attend
Plot what MiniGPT's first layer attends to (code below). Rows can only look left — that's the causal mask!

In [ ]:
sample = "humpty dumpty sat on a wall."
x = torch.tensor([[stoi[c] for c in sample]], device=device)
layer = gpt.blocks.layers[0]
with torch.no_grad():
    h = layer.norm1(gpt.tok(x) + gpt.pos(torch.arange(x.shape[1], device=device)))
    T = x.shape[1]
    _, att = layer.self_attn(h, h, h, attn_mask=nn.Transformer.generate_square_subsequent_mask(T).to(device),
                             need_weights=True)
plt.figure(figsize=(7, 7)); plt.imshow(att[0].cpu(), cmap="Blues")
plt.xticks(range(T), list(sample)); plt.yticks(range(T), list(sample)); plt.title("MiniGPT layer 1 (average of heads)")
plt.show()

## Part 5: Encoder vs decoder — BERT vs GPT 🔀
The original 2017 Transformer had two halves. Today's models usually use one:

| | **Encoder** (e.g. BERT) | **Decoder** (e.g. GPT, Claude) |
|---|---|---|
| Looks at | the **whole** sentence, both directions | only **earlier** words (causal mask) |
| Good at | understanding: fill in blanks, classify, search | **generating** text word by word |
| Analogy | A reader who sees the whole page | A writer who can't see words they haven't written yet |

Let's try both:

In [ ]:
from transformers import pipeline
encoder = pipeline("fill-mask", model="bert-base-uncased")
decoder = pipeline("text-generation", model="distilgpt2")

print("ENCODER fills a blank using words on BOTH sides:")
for r in encoder("The [MASK] barked loudly at the postman.")[:3]:
    print(f"   {r['token_str']:8} {r['score']:.0%}")

print("\nDECODER continues from the LEFT only:")
print("  ", decoder("The dog barked loudly at", max_new_tokens=15, do_sample=True)[0]["generated_text"])

### 🧩 Exercise 3
1. Give the encoder a sentence where the clue comes **after** the blank: `"The [MASK] meowed and chased the mouse."` Why can't a decoder do this as easily?
2. Which type would you choose for: (a) a homework chatbot, (b) a spam filter, (c) a search engine, (d) writing song lyrics?

## 📌 Summary
- **Self-attention:** Query × Key → scores → softmax → mix of Values; scale by √d
- **Positional encoding** tells the model word order
- **Multi-head attention** = several attention "detectives" in parallel
- **Transformer block** = attention + feed-forward network + residuals + layer norm, stacked many times
- **Encoders** (BERT) understand; **decoders** (GPT, Claude) generate, using a **causal mask**